# Assignment 2 — Build Simple VGG and ResNet
**YunSeok Choi** | ys.choi@skku.edu | Sungkyunkwan University

Implement simplified **VGG-11** and **ResNet-18** for **CIFAR-10** using PyTorch,
train them, and compare their performance.

## 0. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Load CIFAR-10 Dataset

In [ ]:
# Standard CIFAR-10 normalization (per-channel mean / std)
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_dataset = datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
test_dataset  = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=2)

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f'Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}')

In [ ]:
# Quick visualisation (de-normalize for display)
def denorm(img):
    img = img.numpy().transpose(1, 2, 0)
    img = img * np.array(CIFAR_STD) + np.array(CIFAR_MEAN)
    return np.clip(img, 0, 1)

# use a non-augmented view just for display
disp = datasets.CIFAR10(root='./data', train=True, download=False, transform=test_transform)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = disp[i]
    ax.imshow(denorm(img))
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis('off')
plt.suptitle('CIFAR-10 Samples', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Model Definition — VGG-11

A simplified VGG-11 adapted for 32x32 CIFAR-10 images.
The original VGG uses five maxpool stages designed for 224x224 inputs; here we
keep the same conv configuration but the final feature map becomes 1x1 for a
32x32 input, so a single Linear classifier head is enough (no 4096-d FC stack).

In [ ]:
# VGG-11 configuration: number = output channels, 'M' = maxpool
VGG11_CFG = [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']

class VGG(nn.Module):
    def __init__(self, cfg=VGG11_CFG, num_classes=10, use_batchnorm=True):
        super().__init__()
        self.features = self._make_layers(cfg, use_batchnorm)
        # After 5 maxpools: 32 -> 1, so feature map is 512 x 1 x 1
        self.classifier = nn.Linear(512, num_classes)

    def _make_layers(self, cfg, use_batchnorm):
        layers = []
        in_channels = 3
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                layers.append(nn.Conv2d(in_channels, v, kernel_size=3, padding=1))
                if use_batchnorm:
                    layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(inplace=True))
                in_channels = v
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

## 3. Model Definition — ResNet-18

A simplified ResNet-18 adapted for CIFAR-10. The first layer is a 3x3 conv with
stride 1 (instead of the original 7x7 stride-2 + maxpool, which would shrink the
small 32x32 image too aggressively). It uses BasicBlocks with identity / 1x1
projection shortcuts.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        # Shortcut: project when shape changes
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ResNet(nn.Module):
    def __init__(self, block=BasicBlock, num_blocks=(2, 2, 2, 2), num_classes=10):
        super().__init__()
        self.in_channels = 64
        # CIFAR stem: 3x3 conv, stride 1 (no initial maxpool)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64,  num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        return self.fc(out)


def ResNet18(num_classes=10):
    return ResNet(BasicBlock, (2, 2, 2, 2), num_classes)

## 4. Training & Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        out  = model(X)
        total_loss += criterion(out, y).item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def run_experiment(model, epochs=30, lr=0.1, weight_decay=5e-4, use_scheduler=True):
    """Train a given model and return its training history."""
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=weight_decay)
    scheduler = None
    if use_scheduler:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}
    for epoch in range(1, epochs + 1):
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer)
        vl, va = evaluate(model, test_loader, criterion)
        if scheduler:
            scheduler.step()
        history['train_loss'].append(tl)
        history['train_acc'].append(ta)
        history['test_loss'].append(vl)
        history['test_acc'].append(va)
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:02d} | Train Acc {ta:.4f} | Test Acc {va:.4f} | Test Loss {vl:.4f}')
    return model, history


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 5. Experiment 1 — Train VGG-11

> **Goal**: train the simplified VGG-11 on CIFAR-10 and record its curves.

In [ ]:
vgg = VGG(use_batchnorm=True)
print(f'VGG-11 trainable params: {count_params(vgg):,}')

print('\n=== Training VGG-11 ===')
vgg, vgg_hist = run_experiment(vgg, epochs=30)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(vgg_hist['train_loss'], label='Train')
axes[0].plot(vgg_hist['test_loss'],  label='Test')
axes[0].set_title('VGG-11 Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(vgg_hist['train_acc'], label='Train')
axes[1].plot(vgg_hist['test_acc'],  label='Test')
axes[1].set_title('VGG-11 Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True)

plt.suptitle('Experiment 1: VGG-11 Training Curves')
plt.tight_layout()
plt.show()

print(f'\nVGG-11 Best Test Accuracy: {max(vgg_hist["test_acc"]):.4f}')

## 6. Experiment 2 — Train ResNet-18

> **Goal**: train the simplified ResNet-18 on CIFAR-10 and record its curves.

In [ ]:
resnet = ResNet18()
print(f'ResNet-18 trainable params: {count_params(resnet):,}')

print('\n=== Training ResNet-18 ===')
resnet, res_hist = run_experiment(resnet, epochs=30)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(res_hist['train_loss'], label='Train')
axes[0].plot(res_hist['test_loss'],  label='Test')
axes[0].set_title('ResNet-18 Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(res_hist['train_acc'], label='Train')
axes[1].plot(res_hist['test_acc'],  label='Test')
axes[1].set_title('ResNet-18 Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True)

plt.suptitle('Experiment 2: ResNet-18 Training Curves')
plt.tight_layout()
plt.show()

print(f'\nResNet-18 Best Test Accuracy: {max(res_hist["test_acc"]):.4f}')

## 7. Comparison — VGG-11 vs. ResNet-18

> **Goal**: compare the two architectures directly on test accuracy / loss and
> discuss the effect of residual connections.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(vgg_hist['test_acc'], label='VGG-11')
axes[0].plot(res_hist['test_acc'], label='ResNet-18')
axes[0].set_title('Test Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(vgg_hist['test_loss'], label='VGG-11')
axes[1].plot(res_hist['test_loss'], label='ResNet-18')
axes[1].set_title('Test Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True)

plt.suptitle('Experiment 3: VGG-11 vs. ResNet-18')
plt.tight_layout()
plt.show()

print('--- Summary ---')
print(f'  VGG-11    | params: {count_params(VGG()):>10,} | best test acc: {max(vgg_hist["test_acc"]):.4f}')
print(f'  ResNet-18 | params: {count_params(ResNet18()):>10,} | best test acc: {max(res_hist["test_acc"]):.4f}')